In [ ]:
!pip install torch matplotlib transformers

In [ ]:
import torch
import random
import pandas as pd
from tqdm.auto import tqdm
from datasets import Dataset as HFDataset   # para leer el DataFrame
from torch.utils.data import Dataset as TorchDataset, DataLoader
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    set_seed
)
from peft import LoraConfig, get_peft_model

seed = 44
torch.manual_seed(seed)
random.seed(seed)
set_seed(seed)

import transformers
print("Transformers version:", transformers.__version__)

Transformers version: 4.57.1


In [ ]:
import transformers
print(transformers.__version__)

4.57.1


In [ ]:
from huggingface_hub import login
TOKEN = ''
login(token=TOKEN)

In [ ]:
model_name = "google/gemma-3-1b-it"

tokenizer = AutoTokenizer.from_pretrained(model_name)

# Si no tiene pad_token, usar eos_token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

model.config.pad_token_id = tokenizer.pad_token_id

In [ ]:

!pip install -q transformers accelerate bitsandbytes peft

In [ ]:
tokenizer.pad_token_id

0

In [ ]:
dtypes = {param.dtype for param in model.parameters()}
dtypes

{torch.bfloat16}

In [ ]:
ds_train

DatasetDict({
    train: Dataset({
        features: ['codigo_comun', 'non_pls', 'pls'],
        num_rows: 3545
    })
    test: Dataset({
        features: ['codigo_comun', 'non_pls', 'pls'],
        num_rows: 18
    })
})

In [ ]:
base_prompt = """Using the next instructions, write a plain-language summary for patients:
* Organize it using short, reader-friendly headings similar to those used in patient-oriented evidence summaries (e.g., Review question, Background, Study characteristics, Key results, Conclusions, Quality of evidence).
* Do not use the technical headings verbatim; instead, adapt them into simple, descriptive labels.
* Integrate all information into a clear, coherent, and easy-to-follow narrative.
* Keep the language at or below a 6th-grade reading level.
* Avoid jargon; if you must use a technical term, explain it in simple, familiar words.
* Use active voice, mostly short words (one or two syllables), and sentences of no more than 20 words.
* Organize the summary into short paragraphs of 3-5 sentences
* Use simple numbers or ratios (for example, 1 in 2) instead of percentages.
* Do not add, remove, or infer any information not present in the abstract.
* The summary should remain consistent regardless of the order of sentences in the original abstract.
* Provide only the summary as plain text. Do not use bold, italics, headings, asterisks, or any other formatting
* Do not provide comments or explanations.
* Target a total length of 300–500 words.
Here is the abstract of a biomedical study to summarize:"""

In [ ]:
def format_sample(sample, base_prompt):
    '''
    Función para crear prompt a partir de un texto técnico
    '''
    non_pls = sample["non_pls"]

    prompt = (
        f"{base_prompt}\n\n"
        f"{non_pls}\n\n"
        "Plain-language summary:\n\n"
    )
    return prompt

In [ ]:
class QADataset(TorchDataset):
    def __init__(self, dataset, tokenizer, base_prompt, max_length=2048):
        self.dataset = dataset
        self.tokenizer = tokenizer
        self.base_prompt = base_prompt
        self.max_length = max_length

    def normalize_text(self, value):
        """Convierte cualquier tipo (list, float, int, NaN, None) a string válida."""
        if value is None:
            return ""
        if isinstance(value, float):
            if pd.isna(value):
                return ""
            return str(value)
        if isinstance(value, int):
            return str(value)
        if isinstance(value, list):
            return " ".join(str(x) for x in value)
        return str(value)

    def safe_tokenize(self, text, max_len):
        """Siempre devuelve una lista de int, nunca un int suelto."""
        enc = self.tokenizer(
            text,
            truncation=True,
            max_length=max_len,
            add_special_tokens=False,
        )
        ids = enc.get("input_ids", [])
        if isinstance(ids, int):
            ids = [ids]
        if ids is None:
            ids = []
        return ids

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        sample = self.dataset[idx]

        # Normalizar textos
        non_pls = self.normalize_text(sample["non_pls"])
        pls = self.normalize_text(sample["pls"])

        # Si el texto está vacío, poner algo
        if non_pls.strip() == "":
            non_pls = "[EMPTY]"

        if pls.strip() == "":
            pls = "[EMPTY]"

        # ----- PROMPT -----
        prompt_text = format_sample({"non_pls": non_pls}, self.base_prompt)
        max_prompt_len = self.max_length // 2

        prompt_ids = self.safe_tokenize(prompt_text, max_prompt_len)

        # ----- TARGET -----
        target_text = pls + self.tokenizer.eos_token
        max_target_len = self.max_length - len(prompt_ids)

        target_ids = self.safe_tokenize(target_text, max_target_len)

        # ----- CONCAT -----
        input_ids = prompt_ids + target_ids
        labels = [-100] * len(prompt_ids) + target_ids
        attention_mask = [1] * len(input_ids)

        # Seguridad adicional
        if not isinstance(input_ids, list):
            input_ids = [self.tokenizer.pad_token_id]
            labels = [-100]
            attention_mask = [0]

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels
        }


In [ ]:
df = pd.read_excel("archivos_comparados.xlsx")
ds_full = HFDataset.from_pandas(df)
ds_train = ds_full.train_test_split(test_size=0.005, seed=seed)

train_data = QADataset(ds_train["train"], tokenizer, base_prompt)
eval_data  = QADataset(ds_train["test"],  tokenizer, base_prompt)

In [ ]:
eval_data

In [ ]:
def custom_collate_fn(batch):
    max_length = max(len(x["input_ids"]) for x in batch)

    input_ids = []
    attention_masks = []
    labels = []

    for item in batch:
        ids = item["input_ids"]
        mask = item["attention_mask"]
        lbls = item["labels"]

        pad_len = max_length - len(ids)

        padded_input_ids = ids + [tokenizer.pad_token_id] * pad_len
        padded_attention_mask = mask + [0] * pad_len
        padded_labels = lbls + [-100] * pad_len

        input_ids.append(padded_input_ids)
        attention_masks.append(padded_attention_mask)
        labels.append(padded_labels)

    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attention_masks, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
    }


In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj","k_proj","v_proj","o_proj"],
    bias="none",
    lora_dropout=0.05,
    task_type="CAUSAL_LM"
)

# Se obtiene el modelo con LoRA
model = get_peft_model(model, lora_config)

In [ ]:
training_args = TrainingArguments(
    output_dir="./gemma3_lora_pls",
    per_device_train_batch_size=1,       # más seguro para VRAM
    gradient_accumulation_steps=8,
    per_device_eval_batch_size=1,
    learning_rate=3e-5,
    #max_steps=1000,
    num_train_epochs=5,
    bf16=True,
    eval_strategy="epoch",
    eval_steps=100,
    logging_steps=50,
    save_steps=100,
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none",
    save_total_limit=1,
    seed=seed,
    data_seed=seed,
    metric_for_best_model="eval_loss"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=eval_data,
    data_collator=custom_collate_fn,
)

The model is already on multiple devices. Skipping the move to device specified in `args`.


In [ ]:
import gc
gc.collect()

1379

In [ ]:
from torch.utils.data import DataLoader

train_data = QADataset(ds_train['train'], tokenizer, base_prompt)
dl = DataLoader(train_data, batch_size=2, collate_fn=custom_collate_fn)

batch = next(iter(dl))
for k, v in batch.items():
    print(k, v.shape)

input_ids torch.Size([2, 2048])
attention_mask torch.Size([2, 2048])
labels torch.Size([2, 2048])


In [ ]:
sample = train_data[0]
print(sample.keys(), len(sample["input_ids"]))

from torch.utils.data import DataLoader
dl = DataLoader(train_data, batch_size=2, collate_fn=custom_collate_fn)
batch = next(iter(dl))
for k, v in batch.items():
    print(k, v.shape)


dict_keys(['input_ids', 'attention_mask', 'labels']) 2048
input_ids torch.Size([2, 2048])
attention_mask torch.Size([2, 2048])
labels torch.Size([2, 2048])


In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,1.780600,1.799880
2,1.752500,1.792109
3,1.752400,1.786199
4,1.761200,1.783171
5,1.714500,1.781936


TrainOutput(global_step=2220, training_loss=1.743774128819371, metrics={'train_runtime': 4380.8928, 'train_samples_per_second': 4.046, 'train_steps_per_second': 0.507, 'total_flos': 1.1703817050966144e+17, 'train_loss': 1.743774128819371, 'epoch': 5.0})

In [ ]:
save_dir = "gemma-3-1b-it"
model.save_pretrained(save_dir)

In [ ]:
import shutil

folder_to_download = "gemma3_lora_pls"

# Comprimir a ZIP
shutil.make_archive(folder_to_download, 'zip', folder_to_download)

'/content/gemma3_lora_pls.zip'

In [ ]:
folder_to_download = "gemma-3-1b-it"

# Comprimir a ZIP
shutil.make_archive(folder_to_download, 'zip', folder_to_download)

'/content/gemma-3-1b-it.zip'

In [ ]:
from google.colab import files
files.download("gemma-3-1b-it.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files
files.download("gemma3_lora_pls.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>